<a href="https://colab.research.google.com/github/YashLohchab/house-price-prediction-svm-svc/blob/main/House_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install imbalanced-learn -q

In [1]:
from google.colab import files
uploaded = files.upload()

Saving kc_house_data.csv to kc_house_data.csv


In [2]:
"""
House Sales Prediction and Classification
Kaggle Dataset: House Sales Prediction and Classification (King County, WA)
- Handles class imbalance via SMOTE
- SVM regression (with SGDRegressor for loss curves) and classification
- Loss curves for training visualization
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVR, SVC
from sklearn.linear_model import SGDRegressor, SGDClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

# SMOTE for handling imbalanced classification data
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    SMOTE_AVAILABLE = True
except ImportError:
    print("[INFO] imbalanced-learn not found. Installing...")
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "imbalanced-learn", "--break-system-packages", "-q"])
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    SMOTE_AVAILABLE = True

In [3]:
# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
print("=" * 60)
print("  HOUSE SALES PREDICTION & CLASSIFICATION")
print("=" * 60)

# Try to load from uploads, else generate synthetic King-County-like data
import os
UPLOAD_PATH = "/content"
CSV_NAME = None

for f in os.listdir(UPLOAD_PATH) if os.path.exists(UPLOAD_PATH) else []:
    if f.endswith(".csv"):
        CSV_NAME = os.path.join(UPLOAD_PATH, f)
        break

if CSV_NAME:
    print(f"\n[DATA] Loading uploaded file: {CSV_NAME}")
    df = pd.read_csv(CSV_NAME)
else:
    print("\n[DATA] No CSV found — generating synthetic King County–style data...")
    np.random.seed(42)
    n = 5000

    bedrooms    = np.random.choice([1, 2, 3, 4, 5, 6], n, p=[0.05, 0.20, 0.35, 0.25, 0.10, 0.05])
    bathrooms   = np.round(np.random.uniform(1, 4, n) * 2) / 2
    sqft_living = np.random.randint(500, 6000, n)
    sqft_lot    = np.random.randint(1000, 40000, n)
    floors      = np.random.choice([1, 1.5, 2, 2.5, 3], n, p=[0.35, 0.10, 0.35, 0.10, 0.10])
    waterfront  = np.random.choice([0, 1], n, p=[0.93, 0.07])
    view        = np.random.choice([0, 1, 2, 3, 4], n, p=[0.70, 0.05, 0.10, 0.10, 0.05])
    condition   = np.random.choice([1, 2, 3, 4, 5], n, p=[0.03, 0.07, 0.60, 0.20, 0.10])
    grade       = np.random.choice(range(3, 14), n)
    sqft_above  = sqft_living - np.random.randint(0, 500, n).clip(0, sqft_living - 200)
    sqft_basement = sqft_living - sqft_above
    yr_built    = np.random.randint(1900, 2015, n)
    yr_renovated = np.where(np.random.rand(n) < 0.05, np.random.randint(1990, 2015, n), 0)

    noise = np.random.normal(0, 30000, n)
    price = (
        100 * sqft_living
        + 50000 * bedrooms
        + 30000 * bathrooms
        + 200000 * waterfront
        + 20000 * view
        + 15000 * grade
        + noise
    ).clip(80000, 2000000)

    df = pd.DataFrame({
        "price": price.astype(int),
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "sqft_living": sqft_living,
        "sqft_lot": sqft_lot,
        "floors": floors,
        "waterfront": waterfront,
        "view": view,
        "condition": condition,
        "grade": grade,
        "sqft_above": sqft_above,
        "sqft_basement": sqft_basement,
        "yr_built": yr_built,
        "yr_renovated": yr_renovated,
    })

print(f"  Rows: {len(df):,}  |  Columns: {df.shape[1]}")
print(df.head(3))


  HOUSE SALES PREDICTION & CLASSIFICATION

[DATA] Loading uploaded file: /content/kc_house_data.csv
  Rows: 21,613  |  Columns: 21
           id             date     price  bedrooms  bathrooms  sqft_living  \
0  7129300520  20141013T000000  221900.0         3       1.00         1180   
1  6414100192  20141209T000000  538000.0         3       2.25         2570   
2  5631500400  20150225T000000  180000.0         2       1.00          770   

   sqft_lot  floors  waterfront  view  ...  grade  sqft_above  sqft_basement  \
0      5650     1.0           0     0  ...      7        1180              0   
1      7242     2.0           0     0  ...      7        2170            400   
2     10000     1.0           0     0  ...      6         770              0   

   yr_built  yr_renovated  zipcode      lat     long  sqft_living15  \
0      1955             0    98178  47.5112 -122.257           1340   
1      1951          1991    98125  47.7210 -122.319           1690   
2      1933           

In [4]:
print("\n[PREPROCESSING] Cleaning data...")

# Drop id/date if present
for col in ["id", "date","lat","long",'zipcode']:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# Fill numeric NaNs
df.fillna(df.median(numeric_only=True), inplace=True)

#df = pd.get_dummies(df, columns=['city'])

# Target
TARGET = "price"
FEATURES = [c for c in df.columns if c != TARGET]

X = df[FEATURES].copy()
y_reg = df[TARGET].copy()


[PREPROCESSING] Cleaning data...


In [6]:
df.columns

Index(['price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
       'waterfront', 'view', 'condition', 'grade', 'sqft_above',
       'sqft_basement', 'yr_built', 'yr_renovated', 'sqft_living15',
       'sqft_lot15'],
      dtype='object')

In [7]:
q33 = y_reg.quantile(0.33)
q66 = y_reg.quantile(0.66)
print(f"\n[CLASS] Price thresholds  Low <{q33:,.0f}  |  Mid <{q66:,.0f}  |  High ≥{q66:,.0f}")

y_cls = pd.cut(
    y_reg,
    bins=[-np.inf, q33, q66, np.inf],
    labels=["Low", "Mid", "High"]
)
# Make High slightly rarer to simulate imbalance
mask_high = y_cls == "High"
drop_idx  = y_cls[mask_high].sample(frac=0.55, random_state=42).index
df_bal = df.drop(index=drop_idx).reset_index(drop=True)
X_cls  = df_bal[FEATURES].copy()
y_cls2 = pd.cut(
    df_bal[TARGET],
    bins=[-np.inf, q33, q66, np.inf],
    labels=["Low", "Mid", "High"]
)
le = LabelEncoder()
y_cls_enc = le.fit_transform(y_cls2)

print(f"  Class distribution (before SMOTE): {dict(zip(le.classes_, np.bincount(y_cls_enc)))}")




[CLASS] Price thresholds  Low <360,000  |  Mid <560,000  |  High ≥560,000
  Class distribution (before SMOTE): {'High': np.int64(3283), 'Low': np.int64(7226), 'Mid': np.int64(7091)}


In [8]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls_enc, test_size=0.2, random_state=42, stratify=y_cls_enc)


In [9]:
scaler_r = StandardScaler()
X_train_r_s = scaler_r.fit_transform(X_train_r)
X_test_r_s  = scaler_r.transform(X_test_r)

# Scale the target for SVR
from sklearn.preprocessing import MinMaxScaler
y_scaler = MinMaxScaler()
y_train_r_s = y_scaler.fit_transform(y_train_r.values.reshape(-1, 1)).ravel()
y_test_r_s  = y_scaler.transform(y_test_r.values.reshape(-1, 1)).ravel()

scaler_c = StandardScaler()
X_train_c_s = scaler_c.fit_transform(X_train_c)
X_test_c_s  = scaler_c.transform(X_test_c)


In [10]:
print("\n[SMOTE] Applying SMOTE to classification training set...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_c_sm, y_train_c_sm = smote.fit_resample(X_train_c_s, y_train_c)
print(f"  After SMOTE: {dict(zip(le.classes_, np.bincount(y_train_c_sm)))}")




[SMOTE] Applying SMOTE to classification training set...
  After SMOTE: {'High': np.int64(5781), 'Low': np.int64(5781), 'Mid': np.int64(5781)}


In [13]:
print("\n[SVR] Training Support Vector Regressor (RBF kernel)...")
# Use a sample for SVR (O(n²) memory)
SAMPLE = min(3000, len(X_train_r_s))
idx_s  = np.random.choice(len(X_train_r_s), SAMPLE, replace=False)
svr = SVR(kernel="rbf", C=1, epsilon=0.01, gamma=0.01)
svr.fit(X_train_r_s[idx_s], y_train_r_s[idx_s])

y_pred_svr_s = svr.predict(X_test_r_s)
y_pred_svr   = y_scaler.inverse_transform(y_pred_svr_s.reshape(-1, 1)).ravel()
mae_svr  = mean_absolute_error(y_test_r, y_pred_svr)
rmse_svr = np.sqrt(mean_squared_error(y_test_r, y_pred_svr))
r2_svr   = r2_score(y_test_r, y_pred_svr)
print(f"  MAE: ${mae_svr:,.0f}  |  RMSE: ${rmse_svr:,.0f}  |  R²: {r2_svr:.4f}")



[SVR] Training Support Vector Regressor (RBF kernel)...
  MAE: $127,818  |  RMSE: $207,026  |  R²: 0.7165


In [22]:
print("\n[SVC] Training Support Vector Classifier...")
svc = SVC(kernel="rbf", C=50, gamma=0.1, class_weight="balanced", random_state=42)
svc.fit(X_train_c_sm, y_train_c_sm)

y_pred_svc = svc.predict(X_test_c_s)
print("\n  Classification Report (SVC):")
print(classification_report(y_test_c, y_pred_svc, target_names=le.classes_))



[SVC] Training Support Vector Classifier...

  Classification Report (SVC):
              precision    recall  f1-score   support

        High       0.62      0.68      0.65       657
         Low       0.71      0.71      0.71      1445
         Mid       0.60      0.57      0.59      1418

    accuracy                           0.65      3520
   macro avg       0.64      0.66      0.65      3520
weighted avg       0.65      0.65      0.65      3520



In [15]:
print("\n[LOSS CURVES] Training SGD models (SVM loss) to capture learning curves...")

EPOCHS  = 80
BATCH   = 256

# --- Regression loss curve ---
sgd_reg  = SGDRegressor(loss="epsilon_insensitive", epsilon=0.01,
                         learning_rate="invscaling", eta0=0.01,
                         random_state=42, max_iter=1)
train_losses_r, val_losses_r = [], []
for epoch in range(EPOCHS):
    idx_ep = np.random.permutation(len(X_train_r_s))
    for start in range(0, len(idx_ep), BATCH):
        batch = idx_ep[start:start + BATCH]
        sgd_reg.partial_fit(X_train_r_s[batch], y_train_r_s[batch])
    tr_pred = sgd_reg.predict(X_train_r_s)
    va_pred = sgd_reg.predict(X_test_r_s)
    # convert back to dollars via inverse transform
    tr_mae = mean_absolute_error(y_train_r_s, tr_pred)
    va_mae = mean_absolute_error(y_test_r_s,  va_pred)
    train_losses_r.append(tr_mae)
    val_losses_r.append(  va_mae)

# --- Classification loss curve (hinge = SVM) ---
from sklearn.utils.class_weight import compute_class_weight
cw_vals = compute_class_weight("balanced",
                                classes=np.unique(y_train_c_sm),
                                y=y_train_c_sm)
cw_dict = dict(enumerate(cw_vals))
sgd_cls  = SGDClassifier(loss="hinge", learning_rate="optimal",
                          class_weight=cw_dict, random_state=42, max_iter=1)
train_losses_c, val_losses_c = [], []
for epoch in range(EPOCHS):
    idx_ep = np.random.permutation(len(X_train_c_sm))
    for start in range(0, len(idx_ep), BATCH):
        batch = idx_ep[start:start + BATCH]
        sgd_cls.partial_fit(
            X_train_c_sm[batch], y_train_c_sm[batch],
            classes=np.unique(y_train_c_sm)
        )
    tr_pred_c = sgd_cls.predict(X_train_c_sm)
    va_pred_c = sgd_cls.predict(X_test_c_s)
    tr_acc = np.mean(tr_pred_c == y_train_c_sm)
    va_acc = np.mean(va_pred_c == y_test_c)
    train_losses_c.append(1 - tr_acc)
    val_losses_c.append(  1 - va_acc)

print(f"  Final val regression MAE (×10⁵): {val_losses_r[-1]:.4f}")
print(f"  Final val classification error : {val_losses_c[-1]:.4f}")



[LOSS CURVES] Training SGD models (SVM loss) to capture learning curves...
  Final val regression MAE (×10⁵): 0.0185
  Final val classification error : 0.4688


In [16]:
print("\n[LEARNING CURVES] Computing sklearn learning curves (SVR sample)...")
from sklearn.svm import SVR as _SVR
lc_sizes = np.linspace(0.1, 1.0, 6)
# Use a lighter SVR for speed
lc_pipe = SVR(kernel="rbf", C=50, epsilon=15000, gamma="scale")
lc_sample = min(2000, len(X_train_r_s))
idx_lc = np.random.choice(len(X_train_r_s), lc_sample, replace=False)

train_sizes, train_scores, val_scores = learning_curve(
    lc_pipe,
    X_train_r_s[idx_lc], y_train_r.values[idx_lc],
    train_sizes=lc_sizes,
    scoring="neg_mean_absolute_error",
    cv=3, n_jobs=-1
)
lc_train_mean = -train_scores.mean(axis=1) / 1e5
lc_val_mean   = -val_scores.mean(axis=1)   / 1e5
lc_train_std  = train_scores.std(axis=1)   / 1e5
lc_val_std    = val_scores.std(axis=1)     / 1e5



[LEARNING CURVES] Computing sklearn learning curves (SVR sample)...


In [17]:
print("\n[PLOT] Generating figures...")

plt.rcParams.update({
    "figure.facecolor": "#0f1117",
    "axes.facecolor":   "#1a1d27",
    "axes.edgecolor":   "#3a3d4d",
    "axes.labelcolor":  "#e0e0e0",
    "xtick.color":      "#a0a0a0",
    "ytick.color":      "#a0a0a0",
    "text.color":       "#e0e0e0",
    "grid.color":       "#2a2d3d",
    "grid.linestyle":   "--",
    "grid.alpha":       0.6,
    "legend.facecolor": "#1a1d27",
    "legend.edgecolor": "#3a3d4d",
    "font.family":      "monospace",
})

BLUE  = "#4f9cf9"
TEAL  = "#00d4aa"
ORANGE= "#ff7043"
PURPLE= "#b57bee"
GRAY  = "#6c7080"

fig = plt.figure(figsize=(20, 22))
fig.suptitle("House Sales Prediction & Classification  ·  SVM Analysis",
             fontsize=18, fontweight="bold", color="#ffffff", y=0.98)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Plot 1: Regression loss curve (SGD/SVM) ──────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(train_losses_r, color=BLUE,   lw=2, label="Train Loss")
ax1.plot(val_losses_r,   color=ORANGE, lw=2, label="Val Loss",   ls="--")
ax1.set_title("SVM Regression · Loss Curve", fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Normalized MAE (0–1 scale)")
ax1.legend(); ax1.grid(True)

# ── Plot 2: Classification loss curve (SGD/Hinge) ────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(train_losses_c, color=TEAL,   lw=2, label="Train Error")
ax2.plot(val_losses_c,   color=PURPLE, lw=2, label="Val Error",  ls="--")
ax2.set_title("SVM Classifier · Loss Curve (Hinge)", fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Classification Error (1 − Acc)")
ax2.legend(); ax2.grid(True)

# ── Plot 3: sklearn Learning Curve (SVR) ─────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(train_sizes, lc_train_mean, color=BLUE,   lw=2, label="Training MAE")
ax3.plot(train_sizes, lc_val_mean,   color=ORANGE, lw=2, label="Validation MAE", ls="--")
ax3.fill_between(train_sizes,
                 lc_train_mean - lc_train_std,
                 lc_train_mean + lc_train_std, alpha=0.2, color=BLUE)
ax3.fill_between(train_sizes,
                 lc_val_mean - lc_val_std,
                 lc_val_mean + lc_val_std,   alpha=0.2, color=ORANGE)
ax3.set_title("SVR · Learning Curve (vs. Training Size)", fontweight="bold")
ax3.set_xlabel("Training Samples")
ax3.set_ylabel("MAE (×$100K)")
ax3.legend(); ax3.grid(True)

# ── Plot 4: Actual vs Predicted (SVR) ────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.scatter(y_test_r / 1e6, y_pred_svr / 1e6, alpha=0.4,
            color=TEAL, s=15, label="Predictions")
mn = min(y_test_r.min(), y_pred_svr.min()) / 1e6
mx = max(y_test_r.max(), y_pred_svr.max()) / 1e6
ax4.plot([mn, mx], [mn, mx], color=ORANGE, lw=2, label="Ideal")
ax4.set_title(f"SVR · Actual vs Predicted  (R²={r2_svr:.3f})", fontweight="bold")
ax4.set_xlabel("Actual Price ($M)")
ax4.set_ylabel("Predicted Price ($M)")
ax4.legend(); ax4.grid(True)

# ── Plot 5: Residuals ─────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
residuals = y_test_r.values - y_pred_svr
ax5.scatter(y_pred_svr / 1e6, residuals / 1e3, alpha=0.4, color=PURPLE, s=15)
ax5.axhline(0, color=ORANGE, lw=2, ls="--")
ax5.set_title("SVR · Residual Plot", fontweight="bold")
ax5.set_xlabel("Predicted Price ($M)")
ax5.set_ylabel("Residual ($K)")
ax5.grid(True)

# ── Plot 6: Price Distribution ───────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
ax6.hist(y_reg / 1e6, bins=50, color=BLUE, alpha=0.75, edgecolor="#0f1117")
ax6.axvline(q33 / 1e6, color=TEAL,   lw=2, ls="--", label=f"Q33 ${q33/1e6:.2f}M")
ax6.axvline(q66 / 1e6, color=ORANGE, lw=2, ls="--", label=f"Q66 ${q66/1e6:.2f}M")
ax6.set_title("Price Distribution & Class Boundaries", fontweight="bold")
ax6.set_xlabel("Price ($M)")
ax6.set_ylabel("Count")
ax6.legend(); ax6.grid(True)

# ── Plot 7: Confusion Matrix (SVC) ───────────────────────────
ax7 = fig.add_subplot(gs[2, 0])
cm = confusion_matrix(y_test_c, y_pred_svc)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax7, colorbar=False, cmap="Blues")
ax7.set_title("SVC · Confusion Matrix (after SMOTE)", fontweight="bold")
ax7.title.set_color("#e0e0e0")
for text in ax7.texts:
    text.set_color("#000000")

# ── Plot 8: Class Distribution Before/After SMOTE ────────────
ax8 = fig.add_subplot(gs[2, 1])
before = np.bincount(y_train_c)
after  = np.bincount(y_train_c_sm)
x_pos  = np.arange(len(le.classes_))
w = 0.35
ax8.bar(x_pos - w/2, before, width=w, color=ORANGE, alpha=0.85, label="Before SMOTE")
ax8.bar(x_pos + w/2, after,  width=w, color=TEAL,   alpha=0.85, label="After SMOTE")
ax8.set_xticks(x_pos)
ax8.set_xticklabels(le.classes_)
ax8.set_title("Class Imbalance: Before vs After SMOTE", fontweight="bold")
ax8.set_ylabel("Samples")
ax8.legend(); ax8.grid(True, axis="y")

# ── Plot 9: Feature Importance (via linear SVR coefs) ────────
ax9 = fig.add_subplot(gs[2, 2])
from sklearn.svm import LinearSVR
lsvr = LinearSVR(C=0.1, max_iter=5000, random_state=42)
lsvr.fit(X_train_r_s, y_train_r)
coef_abs = np.abs(lsvr.coef_)
feat_imp = pd.Series(coef_abs, index=FEATURES).nlargest(10)
feat_imp.sort_values().plot(kind="barh", ax=ax9, color=PURPLE, edgecolor="#0f1117")
ax9.set_title("Top 10 Features · LinearSVR Coefficients", fontweight="bold")
ax9.set_xlabel("|Coefficient|")
ax9.grid(True, axis="x")

# Final metrics box
metrics_text = (
    f"SVR Results (RBF):\n"
    f"  MAE   = ${mae_svr:>10,.0f}\n"
    f"  RMSE  = ${rmse_svr:>10,.0f}\n"
    f"  R²    = {r2_svr:>12.4f}\n\n"
    f"SVC Accuracy (test):\n"
    f"  {np.mean(y_pred_svc == y_test_c)*100:.1f}%  (balanced classes)"
)
fig.text(0.01, 0.01, metrics_text, fontsize=9, color="#a0ffc0",
         va="bottom", fontfamily="monospace",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="#1a1d27",
                   edgecolor="#3a3d4d", alpha=0.9))

out_path = "/content/house_sales_svm_analysis.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="#0f1117")
plt.close()
print(f"\n[DONE] Figure saved → {out_path}")
print("=" * 60)



[PLOT] Generating figures...

[DONE] Figure saved → /content/house_sales_svm_analysis.png


In [18]:
# Parameter grid
param_grid_svr = {
    'C': [1, 10, 50, 100],
    'gamma': ['scale',0.1, 0.01, 0.001,'auto'],
    'epsilon': [0.001, 0.005, 0.01],
    'kernel': ['rbf']
}

grid_svr = GridSearchCV(
    estimator=SVR(),
    param_grid=param_grid_svr,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=2
)

grid_svr.fit(X_train_r_s[idx_s], y_train_r_s[idx_s])

print("\nBest Parameters for SVR:")
print(grid_svr.best_params_)

svr = grid_svr.best_estimator_

Fitting 5 folds for each of 60 candidates, totalling 300 fits

Best Parameters for SVR:
{'C': 1, 'epsilon': 0.01, 'gamma': 0.01, 'kernel': 'rbf'}


In [19]:
y_pred_svr_s = svr.predict(X_test_r_s)

y_pred_svr = y_scaler.inverse_transform(
    y_pred_svr_s.reshape(-1,1)
).ravel()

mae_svr = mean_absolute_error(y_test_r, y_pred_svr)
rmse_svr = np.sqrt(mean_squared_error(y_test_r, y_pred_svr))
r2_svr = r2_score(y_test_r, y_pred_svr)

print(f"MAE = {mae_svr:,.0f}")
print(f"RMSE = {rmse_svr:,.0f}")
print(f"R² = {r2_svr:.4f}")

MAE = 127,818
RMSE = 207,026
R² = 0.7165


In [20]:
param_grid_svc = {
    'C': [1, 10, 50, 100],
    'gamma': ['scale', 0.1, 0.01, 0.001],
    'kernel': ['rbf']
}

grid_svc = GridSearchCV(
    estimator=SVC(
        class_weight='balanced',
        random_state=42
    ),
    param_grid=param_grid_svc,
    cv=5,
    scoring='accuracy',
    verbose=2,
    n_jobs=-1
)

grid_svc.fit(X_train_c_sm, y_train_c_sm)

print("\nBest Parameters for SVC:")
print(grid_svc.best_params_)

svc = grid_svc.best_estimator_

Fitting 5 folds for each of 16 candidates, totalling 80 fits

Best Parameters for SVC:
{'C': 50, 'gamma': 0.1, 'kernel': 'rbf'}


In [21]:
y_pred_svc = svc.predict(X_test_c_s)

print(classification_report(
    y_test_c,
    y_pred_svc,
    target_names=le.classes_
))

              precision    recall  f1-score   support

        High       0.62      0.68      0.65       657
         Low       0.71      0.71      0.71      1445
         Mid       0.60      0.57      0.59      1418

    accuracy                           0.65      3520
   macro avg       0.64      0.66      0.65      3520
weighted avg       0.65      0.65      0.65      3520

